The DataFrame is Spark's primary high-level API for structured data. It combines the familiarity of a table — named columns, types, SQL-style operations — with the distributed execution model of Spark. Understanding DataFrames deeply is the single biggest skill for the Databricks Associate exam and real-world Spark work.

## The Spark API Hierarchy

![Spark API Hierarchy](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-api-hierarchy.svg)

- **SQL** and **DataFrame/Dataset** both go through Catalyst and Tungsten — they produce identical execution plans
- **RDD** bypasses Catalyst — no automatic optimisation
- In Python, `DataFrame` is the only option; `Dataset[T]` (typed) is Scala/Java only
- `DataFrame` is an alias for `Dataset[Row]`

## DataFrame Structure

A DataFrame is a **distributed table** — it has a schema (column names and types) and rows, partitioned across executors.

![DataFrame Structure](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/dataframe-structure.svg)

Key properties:
- **Immutable** — transformations return a new DataFrame; the original is unchanged
- **Lazily evaluated** — transformations build a plan; actions execute it
- **Schema-enforced** — every row matches the same column types
- **Partitioned** — rows are split across partitions for parallel processing

## The Schema — StructType & StructField

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    LongType, StringType, IntegerType, BooleanType,
    DoubleType, DateType, TimestampType, ArrayType, MapType
)

spark = (
    SparkSession.builder
    .appName("DataFrames")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Define a schema explicitly
schema = StructType([
    StructField("id",     LongType(),    nullable=False),
    StructField("name",   StringType(),  nullable=True),
    StructField("age",    IntegerType(), nullable=True),
    StructField("active", BooleanType(), nullable=True),
])

print(schema)

In [ ]:
# Spark also accepts DDL string syntax — often more concise
schema_ddl = "id LONG NOT NULL, name STRING, age INT, active BOOLEAN"

from pyspark.sql.types import _parse_datatype_string
print(spark.createDataFrame([], schema_ddl).schema)

## Creating DataFrames

In [ ]:
# 1. From a Python list with explicit schema
data = [
    (1, "Alice", 30, True),
    (2, "Bob",   25, False),
    (3, "Carol", 35, True),
    (4, "Dan",   28, True),
    (5, "Eve",   22, False),
]
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

In [ ]:
# 2. Schema inference (convenient, but slower — Spark scans the data)
df_inferred = spark.createDataFrame(data, ["id", "name", "age", "active"])
df_inferred.printSchema()  # types are inferred — may not match your expectations

In [ ]:
# 3. spark.range — fast way to create integer sequences for testing
spark.range(5).show()
spark.range(0, 20, step=4).show()  # start, end, step

In [ ]:
# 4. From a pandas DataFrame
import pandas as pd

pdf = pd.DataFrame({"x": [10, 20, 30], "y": ["a", "b", "c"]})
df_from_pandas = spark.createDataFrame(pdf)
df_from_pandas.show()

## Exploring a DataFrame

In [ ]:
print("Shape       :", df.count(), "rows ×", len(df.columns), "cols")
print("Columns     :", df.columns)
print("dtypes      :", df.dtypes)
print("Partitions  :", df.rdd.getNumPartitions())
df.describe("age").show()   # count, mean, stddev, min, max

## Selecting Columns

There are several ways to reference a column — they are interchangeable but have subtle differences:

In [ ]:
from pyspark.sql.functions import col, expr

# Four equivalent ways to reference a column
df.select("name")           # string name
df.select(col("name"))      # Column object — preferred when applying functions
df.select(df["name"])       # DataFrame attribute — ties the column to this df
df.select(df.name)          # dot syntax — convenient but breaks with spaces/reserved words

# Select multiple columns
df.select("id", "name", "age").show()

# expr() — use SQL expression strings inline
df.select(expr("age + 5 AS age_in_5_years")).show()

## Adding, Renaming, and Dropping Columns

In [ ]:
from pyspark.sql.functions import col, lit, upper, when

# withColumn — add or replace a column
df2 = (
    df
    .withColumn("age_group",  when(col("age") < 30, "under 30").otherwise("30+"))
    .withColumn("name_upper", upper(col("name")))
    .withColumn("constant",   lit(42))          # literal value
)
df2.show()

# withColumnRenamed
df2.withColumnRenamed("name_upper", "NAME").show(truncate=False)

# drop
df2.drop("constant", "name_upper").show()

## Filtering Rows

In [ ]:
# filter() and where() are identical
df.filter(col("age") > 25).show()
df.where("age > 25").show()  # SQL string also works

# Multiple conditions — use & (AND), | (OR), ~ (NOT)
# Always wrap each condition in parentheses!
df.filter((col("age") > 24) & (col("active") == True)).show()
df.filter((col("age") < 25) | (col("age") > 32)).show()

# isin
df.filter(col("name").isin("Alice", "Carol")).show()

# NOT isin
df.filter(~col("name").isin("Alice", "Carol")).show()

## Handling Nulls

In [ ]:
from pyspark.sql.functions import coalesce, col

# Create a DataFrame with nulls
df_nulls = spark.createDataFrame(
    [(1, "Alice", 30), (2, None, None), (3, "Carol", 35)],
    ["id", "name", "age"]
)

# Filter nulls
df_nulls.filter(col("name").isNull()).show()     # rows where name IS NULL
df_nulls.filter(col("name").isNotNull()).show()  # rows where name IS NOT NULL

# Drop rows containing any null
df_nulls.dropna().show()
df_nulls.dropna(subset=["name"]).show()   # only check specific columns

# Fill nulls
df_nulls.fillna({"name": "Unknown", "age": 0}).show()

# coalesce — return first non-null value
df_nulls.select(coalesce(col("name"), lit("N/A")).alias("name_safe")).show()

## Type Casting

In [ ]:
from pyspark.sql.types import DoubleType

# cast() — convert a column's type
df.withColumn("age_double", col("age").cast(DoubleType())).printSchema()

# DDL string form — often more readable
df.withColumn("age_str", col("age").cast("string")).printSchema()

## Sorting and Deduplication

In [ ]:
from pyspark.sql.functions import desc

# orderBy / sort (synonyms)
df.orderBy(col("age").desc()).show()
df.sort("age", ascending=False).show()
df.orderBy(col("active").desc(), col("age").asc()).show()  # multi-column

# distinct — removes duplicate rows (causes a shuffle)
dupes = spark.createDataFrame([(1, "Alice"), (1, "Alice"), (2, "Bob")], ["id", "name"])
dupes.distinct().show()

# dropDuplicates — deduplicate on specific columns
dupes.dropDuplicates(["id"]).show()

## Converting Between APIs

In [ ]:
# DataFrame → pandas (collects all data to driver — only for small results)
pdf = df.toPandas()
print(type(pdf))
print(pdf)

# DataFrame → RDD
rdd = df.rdd          # RDD of Row objects
print(rdd.first())

# RDD → DataFrame
from pyspark.sql import Row
rdd2 = spark.sparkContext.parallelize([
    Row(id=10, name="Zara", age=26, active=True)
])
df_from_rdd = spark.createDataFrame(rdd2)
df_from_rdd.show()

## Summary

- A **DataFrame** is a distributed table with a schema — immutable, lazily evaluated, and partitioned across executors.
- SQL, DataFrame, and Dataset APIs all go through **Catalyst** and produce identical execution plans. RDD bypasses Catalyst.
- Define schemas with `StructType` / `StructField`, or as a DDL string. Explicit schemas are faster and safer than inferred ones.
- Reference columns with strings, `col()`, `df["col"]`, or `expr()` — `col()` is preferred when applying functions.
- `withColumn` adds or replaces; `withColumnRenamed` renames; `drop` removes.
- `filter` / `where` are identical; combine conditions with `&`, `|`, `~` — always parenthesise each condition.
- Handle nulls with `isNull`, `isNotNull`, `dropna`, `fillna`, and `coalesce`.
- `toPandas()` collects to the driver — only use on small DataFrames. `df.rdd` gives the underlying RDD when needed.